# Data Cleaning Pipeline: Titanic Dataset

**Internship Task Submission**
- **Objective**: Profile, deduplicate, clean missing values, normalize formats, validate, and export a clean dataset using Pandas.
- **Dataset**: Titanic passenger records ().

## 1. Data Ingestion & Initial Profiling
We begin by loading the raw dataset and inspecting its structural dimensions, data types, and first few records.

In [1]:
import os
import pandas as pd

# Resolve data path whether running from root or notebooks directory
data_path = "../data/titanic.csv" if os.path.exists("../data/titanic.csv") else "data/titanic.csv" if os.path.exists("data/titanic.csv") else "titanic.csv"
df = pd.read_csv(data_path)
df.head()

### Dataset Dimensions & Schema Audit
We inspect the exact row/column count and the non-null status of each column.

In [2]:
# Total dimensions (rows, columns)
print("Initial shape:", df.shape)

# Overview of columns and data types
df.info()

## 2. Deduplication Audit
Duplicate records can skew statistics and model performance. We check for exact duplicate rows across all columns as well as duplicate candidate keys ().

In [3]:
# Check exact full-row duplicates
print("Full-row duplicates:", df.duplicated().sum())

# Check unique key integrity
print("PassengerId duplicates:", df.duplicated(subset=["PassengerId"]).sum())

# Drop duplicates if any were present
df.drop_duplicates(inplace=True)

## 3. Handling Missing Values
First, we scan every column to identify the number of missing (/) values.

In [4]:
# Missing values per column
df.isnull().sum()

### Missing Value Strategy:
1. ** (687 missing, ~77%)**: Dropping the column because over 75% of values are missing; attempting to impute would introduce massive bias.
2. ** (177 missing, ~20%)**: Imputing with the rounded mean age (30 years) to retain valuable passenger rows without shifting the average.
3. ** (2 missing, ~0.2%)**: Dropping the 2 specific rows with missing boarding ports.

In [5]:
# 1. Drop Cabin column due to excessive missingness
df.drop(columns="Cabin", inplace=True)

# 2. Impute missing Age values with the rounded mean
mean_age = int(df["Age"].mean().round())
df["Age"] = df["Age"].fillna(mean_age)

# 3. Drop rows with missing Embarked values
df = df.dropna(subset=["Embarked"])

# Verify missing values are resolved
print("Remaining missing values:")
print(df.isnull().sum())

## 4. Normalizing Data Formats and Types
We clean and standardize column data types and string values:
- Cast **** to integer () since ages represent whole years and s are now resolved.
- Round **** to 2 decimal places to represent standard currency.
- Strip whitespace and convert text in **** and **** to standard lowercase.

In [6]:
# Type casting and numerical formatting
df["Age"] = df["Age"].astype("int")
df["Fare"] = df["Fare"].round(2)

# Text normalization (trim whitespace and standardize case)
df["Sex"] = df["Sex"].str.strip().str.lower()
df["Embarked"] = df["Embarked"].str.strip().str.lower()

# Preview cleaned types and sample data
df.head()

## 5. Data Validation Checks (Quality Assurance)
Before final export, we validate the dataset to ensure:
- Zero null values remaining.
- Expected row and column counts.
- Correct column data types.
- Value ranges (e.g. no negative fares or impossible ages).

In [7]:
# Validation 1: Zero Nulls
assert df.isnull().sum().sum() == 0, "Validation Error: Null values still present!"
print("✓ Null check passed: 0 missing values.")

# Validation 2: Dimensions check (889 rows, 11 columns)
assert df.shape == (889, 11), f"Validation Error: Unexpected shape {df.shape}"
print(f"✓ Shape check passed: {df.shape[0]} rows, {df.shape[1]} columns.")

# Validation 3: Value sanity checks
assert (df["Age"] > 0).all(), "Validation Error: Non-positive ages found!"
assert (df["Fare"] >= 0).all(), "Validation Error: Negative fares found!"
assert set(df["Sex"].unique()) == {"male", "female"}, "Validation Error: Unexpected Sex values!"
print("✓ Value sanity checks passed.")

# Summary statistics
df.describe()

## 6. Output & Export
We export the cleaned, validated dataset to CSV format (and Parquet if supported), ensuring  so no redundant index column is added.

In [8]:
# Export to CSV
output_csv = "titanic_cleaned.csv"
df.to_csv(output_csv, index=False)
print(f"✓ Successfully exported cleaned dataset to {output_csv}")

# Export to Parquet (if pyarrow/fastparquet available)
try:
    output_parquet = "titanic_cleaned.parquet"
    df.to_parquet(output_parquet, index=False)
    print(f"✓ Successfully exported cleaned dataset to {output_parquet}")
except Exception as e:
    print(f"Note on Parquet: {e}")

## 7. Verification of Exported File
We reload the exported file to confirm that it loads cleanly with identical dimensions and types.

In [9]:
# Reload exported CSV
exported_df = pd.read_csv("titanic_cleaned.csv")
print("Reloaded exported shape:", exported_df.shape)
exported_df.head()